In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import time

# Подготавливаем датасет для загрузки

In [ ]:
data = pd.read_pickle("text_data.pkl")

Собираем все слова из предложений

In [ ]:
all_words = [word.strip() for sentence in data['sentence'] for word in sentence if word.strip() != ""]
all_words = sorted(all_words)

Считаем частоту слов

In [ ]:
word_freq = Counter(all_words)

Создаем словарь из слов с частотой выше указанной

In [ ]:
K = 3  # порог
vocabulary = [word for word, freq in word_freq.items() if freq >= K]

Добавляем специальные символы для пустоты и неизвестного слова

In [ ]:
vocabulary = ['<PAD>', '<UNK>'] + vocabulary

Считаем размер словаря

In [ ]:
vocab_size = len(vocabulary)
print("В словаре :", vocab_size, f"слов (частота ≥ {K})")

В словаре : 15088 слов (частота ≥ 3)


Находим максимальную длину предложения

In [ ]:
data['sentence_len'] = data['sentence'].apply(lambda x: len(x))

Определяем оптимальный размер длины последовательности для передачи в нейросеть

In [ ]:
data['sentence_len'].describe()

,sentence_len
count,30527.000000
mean,15.764700
std,13.652076
min,0.000000
25%,7.000000
50%,12.000000
75%,21.000000
max,231.000000


In [ ]:
max_length = data['sentence'].apply(lambda x: len(x)).max()
print("Максимально длинное предложение состоит из :", max_length, "слов")

Максимально длинное предложение состоит из : 231 слов


In [ ]:
print("95% квантиль длины предложений: ", data['sentence_len'].quantile(0.95))

95% квантиль длины предложений:  42.0


Берем 95% квантиль в качестве длины последовательности для нейросети

In [ ]:
max_length = int(data['sentence_len'].quantile(0.95))

Создаем словарь перевода слова в индексы (порядковые номера в словаре)

In [ ]:
word_to_index = {word: idx for idx, word in enumerate(vocabulary)}
index_to_word = {idx: word for word, idx in word_to_index.items()}

vocab_size = len(vocabulary)
print("Длина словаря: ", vocab_size)

Длина словаря:  15088


Переводим в индексы

In [ ]:
def sentence_to_index(sentence, max_len):
  # Преобразуем предложение в последовательность индексов
  indices = [word_to_index[word.lower()] if word.lower() in vocabulary else word_to_index['<UNK>'] for word in sentence ]

  # Пэддинг до max_len
  if len(indices) < max_len:
      indices += [word_to_index['<PAD>']] * (max_len - len(indices))
  else:
      indices = indices[:max_len]  # обрезаем, если длиннее

  return indices

In [ ]:
data['indices'] = data['sentence'].apply(sentence_to_index, args = (max_length, ))

In [ ]:
data.head()

,sentence,author,sentence_len,indices
0,"[в, начале, июля, в, чрезвычайно, жаркое, врем...",Достоевский,37,"[738, 6429, 4432, 738, 14740, 1, 1511, 8569, 9..."
1,"[он, благополучно, избегнул, встречи, с, своею...",Достоевский,9,"[7490, 392, 1, 1625, 11374, 11542, 14445, 6070..."
2,"[каморка, его, приходилась, под, самою, кровле...",Достоевский,17,"[1, 3163, 10164, 8569, 11410, 1, 1919, 1, 2893..."
3,"[квартирная, же, хозяйка, его, у, которой, он,...",Достоевский,42,"[4611, 3279, 14439, 3163, 13630, 4909, 7490, 6..."
4,"[и, каждый, раз, молодой, человек, проходя, ми...",Достоевский,20,"[4085, 4456, 10758, 5881, 14632, 10500, 5764, ..."


Создаем бинарный таргет

In [ ]:
data['target'] = np.where(data["author"] == "Достоевский", 0, 1)

In [ ]:
data.head()

,sentence,author,sentence_len,indices,target
0,"[в, начале, июля, в, чрезвычайно, жаркое, врем...",Достоевский,37,"[738, 6429, 4432, 738, 14740, 1, 1511, 8569, 9...",0
1,"[он, благополучно, избегнул, встречи, с, своею...",Достоевский,9,"[7490, 392, 1, 1625, 11374, 11542, 14445, 6070...",0
2,"[каморка, его, приходилась, под, самою, кровле...",Достоевский,17,"[1, 3163, 10164, 8569, 11410, 1, 1919, 1, 2893...",0
3,"[квартирная, же, хозяйка, его, у, которой, он,...",Достоевский,42,"[4611, 3279, 14439, 3163, 13630, 4909, 7490, 6...",0
4,"[и, каждый, раз, молодой, человек, проходя, ми...",Достоевский,20,"[4085, 4456, 10758, 5881, 14632, 10500, 5764, ...",0


In [ ]:
data['target'].value_counts()

,count
target,
1,18266
0,12261


In [ ]:
data.to_pickle("vectorized_text_data.pkl")

# Делим на обучающую и тестовую выборки

In [ ]:
X = np.array(data['indices'].tolist())
y = data['target'].values

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 42, stratify = y)

In [ ]:
X_train = torch.tensor(X_train, dtype=torch.long)
X_test = torch.tensor(X_test, dtype=torch.long)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

print(f"Размер train: {len(X_train)}")
print(f"Размер test: {len(X_test)}")

Размер train: 21368
Размер test: 9159


# Создаем DataSet и DataLoader

In [ ]:
class TextDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
# Создаём datasets
train_dataset = TextDataset(X_train, y_train)
test_dataset = TextDataset(X_test, y_test)

# DataLoaders
train_dataloader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False
)

# Собираем RNN с эмбеддингами

In [ ]:
class RNNWithEmbedding(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, output_size, num_layers=1):
        super(RNNWithEmbedding, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx = word_to_index['<PAD>'])
        self.rnn = nn.RNN(embedding_dim, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # x: (batch_size, seq_len) — индексы
        embedded = self.embedding(x)           # (batch_size, seq_len, embedding_dim)
        rnn_out, hidden = self.rnn(embedded)    # rnn_out: (batch_size, seq_len, hidden_size)
        # Берём последний скрытый state
        out = self.fc(hidden[-1]).squeeze(-1)              # (batch_size, output_size)
        return out

# Обучаем RNN

In [ ]:
LEARNING_RATE = 0.0001
EPOCHS = 10
DEVICE = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"

model =  RNNWithEmbedding(vocab_size = vocab_size, embedding_dim = 128, hidden_size = 64, output_size = 1, num_layers = 1)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = LEARNING_RATE)

Обучаем модель

In [ ]:
start_time = time.time()
model.train()
for epoch in range(EPOCHS):
    total_loss = 0
    for X_batch, y_batch in train_dataloader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_dataloader):.4f}")
end_time = time.time()
elapsed_time = end_time - start_time
print(f"Время обучения: {elapsed_time:.4f} секунд")

Epoch 1, Loss: 0.6745
Epoch 2, Loss: 0.6709
Epoch 3, Loss: 0.6691
Epoch 4, Loss: 0.6674
Epoch 5, Loss: 0.6653
Epoch 6, Loss: 0.6624
Epoch 7, Loss: 0.6593
Epoch 8, Loss: 0.6553
Epoch 9, Loss: 0.6511
Epoch 10, Loss: 0.6464
Время обучения: 390.4876 секунд


Сохраняем модель

In [ ]:
torch.save(model.state_dict(), "rnn_embeddings_model.pth")

Применение к тестовому датасету

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

model.eval()  # переводим модель в режим inference

# Собираем истинные метки и предсказания
y_true = []
y_pred = []

with torch.no_grad():
    for X_batch, y_batch in test_dataloader:
        # Получаем логиты (необработанные выходы)
        logits = model(X_batch)
        # Преобразуем в вероятности через сигмоиду
        probs = torch.sigmoid(logits)
        # Бинаризация: порог 0.5
        predicted = (probs > 0.5).float()
        # Сохраняем для метрик
        y_true.extend(y_batch.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())

# Конвертируем в numpy-массивы
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Вычисляем метрики
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)

print(f"Test Accuracy:  {accuracy:.4f}")
print(f"Test Precision: {precision:.4f}")
print(f"Test Recall:    {recall:.4f}")

Test Accuracy:  0.6048
Test Precision: 0.6074
Test Recall:    0.9597


In [ ]:
y_test.mean()

tensor(0.5983)

In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_true, y_pred)
print("\nМатрица ошибок:")
print(cm)


Матрица ошибок:
[[ 280 3399]
 [ 221 5259]]
